<img src="http://imgur.com/1ZcRyrc.png" style="float: left; margin: 20px; height: 55px">

# Introduction to Logistic Regression

_Authors: Kiefer Katovich (SF), Matt Brems (DC)_

---

### Learning Objectives
- Distinguish between regression and classification problems.
- Understand how logistic regression is similar to and different from linear regression.
- Understand the math behind the logit link function (and logistic function).
- Plot the logistic regression for predicting admittance from GPA.
- Understand how to interpret the coefficients of logistic regression.
- Know the benefits of logistic regression as a classifier.

<a id='introduction'></a>

## Introduction

---

Logistic regression is a natural bridge to connect regression and classification.
- Logistic regression is a famous classifier that's still used very, **very** frequently.
- Because it is a regression model, logistic regression will predict continuous values.
    - Logistic regression will predict continuous probabilities between 0 and 1.
    - Example: What is the probability that someone shows up to vote?
- However, logistic regression almost always operates as a classification model.
    - Logistic regression will use these continuous predictions to classify something as 0 or 1.
    - Example: Based on the predicted probability, do we predict that someone votes?

In this lecture, we'll only be reviewing the binary outcome case with two classes, but logistic regression can be generalized to multiple classes.

**Some examples of when logistic regression could be used:**
- Will a user will purchase a product, given characteristics like income, age, and number of family members?
- Given one's scores, GPA, and the prestige of a college, will a student be admitted to a specific college?
- Does this patient have a specific disease, given their symptoms?
- Will a person default on their loan?

And many more. 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import seaborn as sns

plt.style.use('fivethirtyeight')

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

from sklearn.linear_model import LogisticRegression, LinearRegression

<a id='load-data'></a>

## Load the College Admissions Data

---

As we walk through the inner workings of logistic regression, we'll be using some basic college admissions data. The data set only has a few columns:
- `admit`: A binary 1-0 variable indicating whether or not a student was admitted.
- `gre`: The student's GRE score.
- `gpa`: The student's GPA.
- `prestige`: A rating for the college's "prestige."

**Load in the data and perform any necessary cleaning.**

In [ ]:
admissions = pd.read_csv('./datasets/admissions.csv')

In [ ]:
# Check first five rows.


In [ ]:
# Check for missing values.


In [ ]:
# Drop every row that has an NA.


<details><summary>What assumption are we making when we drop rows that have at least one NA in it?</summary>
    
- We assume that what we drop looks like what we have observed. We might say that what we dropped is just a random sample of our whole data. 
- It's not important to know this now, but the formal term is that our data is missing completely at random.
</details>

### Artificially increase the signal in the data.

The signal for detecting admission in the college data is fairly weak — too weak for the purposes of our demonstration. (There are lots of factors to predicting admission other than GPA, test score, and prestige.)

**Obviously you would never do this to your real data**... but in order to make clearer the relationship between our predictors and whether or not someone is admitted, we are making 10 copies of the data set and adding an artificial signal for detecting `admit` to the `gpa` column. 

In [ ]:
# Grab all rows where prestige = 1.
admit = 

# Concatenate ten copies of admit.
admit = 

np.random.seed(42)

admit.loc[admit['admit'] == 1, 'gpa'] += np.random.random(size=admit[admit.admit == 1].shape[0])
admit.loc[admit['admit'] == 0, 'gpa'] -= np.random.random(size=admit[admit.admit == 0].shape[0])

*The addition of random signal will cause slight variance in our results throughout the lesson.*

<a id='expected-value'></a>

## Regression Models Estimate the Expected Value of `y`

---

Logistic regression is a regression. It still constructs a formula with our predictor variables and coefficients to estimate the *expected value* of the target variable.

You're quite familiar with linear regression. We've re-written the formula slightly on the left-hand side:

### $$E(y|\mathbf{X}) = \beta_0 + \beta_1x_1 + \beta_2x_2 + \cdots + \beta_px_p$$

Where:
- $E(y|X)$ is the Expected Value (Average) of $y$, given corresponding predictor values of $\mathbf{X}$.
- $x_1$, $x_2$, $\ldots$, $x_p$ are the predictors $1$ through $p$ (columns) of the $\mathbf{X}$ matrix.
- $\beta_0$ is the intercept.
- $\beta_j$ is the coefficient for the predictor $x_j$, the $j$th column in variable matrix $\mathbf{X}$.


<a id='pred-admit'></a>

## What If We Predicted `admit` With `gpa` Using Linear Regression?

---

Let's try predicting the `admit` binary indicator using just `gpa` with a linear regression to see what goes wrong.

In [ ]:
y = admit['admit']
X = admit[['gpa']] # Because X is usually a matrix, we'll get a 
                   # reshape request if we don't use double brackets.

# Import and do Train/Test Split



# Instantiate linear regression model.


# Fit on training data.


# Print coefficients.
print('Intercept:', )
print('Coef(s):', )

<a id='plot-reg'></a>
### What do those coefficients mean?  Plot `admit` (our $Y$) against `gpa` (our $X$).

Looking at the plot below, what are problems with using a regression?

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))

# Plot the regression line where gpa predicts admit.
x_vals = np.linspace(1.,5.,300)
ax.plot(x_vals, linmod.predict(x_vals[:, np.newaxis]),
        color='black', alpha=0.7, lw=4)

# Create one scatterplot for each type of admit decision (admitted vs. rejected).
ax.scatter(admit['gpa'][admit['admit'] == 0],
           admit['admit'][admit['admit'] == 0],
           c='#FF9E4A', s=100, alpha=0.7,
           label='rejected')

ax.scatter(admit['gpa'][admit['admit'] == 1],
           admit['admit'][admit['admit'] == 1],
           c='#729ECE', s=100, alpha=0.7,
           label='admitted')

ax.set_ylabel('admitted', fontsize=16)
ax.set_xlabel('gpa', fontsize=16)
ax.set_title('admittance ~ gpa, prestige=1', fontsize=20)
# The tilde means admittance follows/depends on GPA. This is syntax seen in patsy or R.

ax.set_xlim([1.5,5])
ax.set_ylim(-0.25, 1.25)

plt.legend(loc='upper left')
plt.show()

#### **Note:** Unless we predict exactly one value, linear regression will always predict values between $-\infty$ and $+\infty$. There are real-world cases when this is inappropriate... because we may want to just make predictions on a smaller set or range of values.

<a id='pred-binary'></a>

## Predicting a Binary Class

---

In our case we have two classes: `1=admitted` and `0=rejected`.

The logistic regression is still solving for an expected value. In our binary classification case, this expected value is the probability of `y` being one of the classes.

### $$E(y) = P(y = 1)$$

As you can see, we still represent this with a regression on the right-hand side.

### $$E(y|X) = P(y = 1|X) = \beta_0 + \sum_{j}^p\beta_jx_j$$

There's a clear problem with this, which we saw in the previous plot. The right side of the equation is in no way bounded to [0, 1]. We need to *guarantee* that the right-hand side of the regression equation will evaluate to a probability estimate.

<a id='logit'></a>

## The Logit Link Function

---

We will use something called a **link function** to effectively "bend" our line of best fit so that it is a curve of best fit that matches the range or set of values in which we're interested.

For logistic regression, that specific link function that transforms ("bends") our line is known as the **logit** link.

### $$\text{logit}\left(P(y = 1)\right) = \beta_0 + \beta_1x_1 + \beta_2x_2 + \cdots + \beta_px_p$$

### $$\ln\left(\frac{P(y = 1)}{1 - P(y = 1)}\right) = \beta_0 + \beta_1x_1 + \beta_2x_2 + \cdots + \beta_px_p$$

Instead of each independent variable $X_i$ being linearly related to $Y$, we assume that each independent variable $X_i$ is linearly related to the **logit of the probability of success**. Equivalently, we assume that each independent variable $X_i$ is linearly related to the **log odds of success**.

Remember, the purpose of the link function bends our line of best fit.
- This is convenient because we can have any values of $X$ inputs that we want, and we'll only ever predict between 0 and 1!
- This is inconvenient, though, because interpreting a one-unit change gets a little harder. (More on this later.)

<a id='logistic-viz'></a>

## Fitting and making predictions with the logistic regression model.

---

In [ ]:
# Step 1: Instantiate our model.


# Step 2: Fit our model.


print('Logreg intercept:', )
print('Logreg coef(s):', )

There are two methods in `sklearn` we will use **a lot** for the rest of this course and beyond:
- `.predict()`
- `.predict_proba()`

In [ ]:
print('Logreg predicted probabilities:', )

In [ ]:
print('Logreg predicted probabilities:', )

<a id='log-odds'></a>

### Using the log-odds —the natural logarithm of the odds.

The combination of converting the "probability of success" to "odds of success," then taking the logarithm of that is called the **logit link function**.

### $$\text{logit}\big(P(y=1)\big) = ln\bigg(\frac{P(y=1)}{1-P(y=1)}\bigg) =  \beta_0 + \beta_1x_1 + \beta_2x_2 + \cdots + \beta_px_p$$

We've bent our line how we want... but how do we interpret our coefficients?

---
<a id='odds-ratios'></a>
### Odds

Probabilities and odds represent the same thing in different ways. The odds for probability **p** is defined as:

### $$\text{odds}(p) = \frac{p}{1-p}$$

The odds of a probability is a measure of how many times more likely it is than the inverse case.

For example, suppose I'm looking at the probability and odds of a specific horse, "Secretariat," winning a race.

- When **`p = 0.5`**: **`odds = 1`**
    - The horse Secretariat is as likely to win as it is to lose.
- When **`p = 0.75`**: **`odds = 3`**
    - The horse Secretariat is three times as likely to win as it is to lose.
- When **`p = 0.40`**: **`odds = 0.666..`**
   - The horse Secretariat is two-thirds as likely to win as it is to lose.

#### Write a function to calculate odds.

In [ ]:
# Plot a range of probabilities 
probabilities = np.linspace(0.005, .99, num=200)
pairs = [(proba, odds(proba)) for proba in probabilities]

# Convert list comprehension into dataframe.
pairs = pd.DataFrame(pairs, columns=["probability", "odds"])

# Plot blue line for odds as probability goes from 0.5% to 99%.
pairs.plot(x="probability", y="odds", figsize=(11, 6))

# Plot red dashed line to visualize odds when probability is 50%.
plt.vlines(.5, 0, 100, linestyles="dashed", color='red')
plt.text(.33, 50, "odds(.5) = 1", fontsize=15, color = 'red')

# Plot orange dashed line to visualize odds when probability is 66.67%.
plt.vlines(.6667, 0, 100, linestyles="dotted", color='orange')
plt.text(.68, 50, "odds(2/3) = 2", fontsize=15, color='orange')

# Annotate blue line when probability is 100%.
plt.text(1, 100, "odds(1) = $\infty$", fontsize=15, color='blue')

# Make title.
plt.title("If your probability of success is 50%, then your odds of success are 1. \nIf your probability of success is 100%, then your odds of success are $\infty$.", ha = 'left', position = (0,1), fontsize=18);

## Interpreting a one-unit change in $X_i$.

$$ln\bigg(\frac{P(y=1)}{1-P(y=1)}\bigg) = \beta_0 + \beta_1x_1 + \beta_2x_2 + \cdots + \beta_px_p$$

Given this model, a one-unit change in $X_i$ implies a $\beta_i$ unit change in the log odds of success.

**This is annoying**.

We often convert log-odds back to "regular odds" when interpreting our coefficient... our mind understands odds better than the log of odds.

So, let's get rid of the log on the left-hand side. Mathematically, we do this by "exponentiating" each side.

$$
\begin{eqnarray*}
ln\bigg(\frac{P(y=1)}{1-P(y=1)}\bigg) &=& \beta_0 + \beta_1x_1 + \beta_2x_2 + \cdots + \beta_px_p \\
\Rightarrow e^{\Bigg\{ln\bigg(\frac{P(y=1)}{1-P(y=1)}\bigg)\Bigg\}} &=& e^{\Bigg\{\beta_0 + \beta_1x_1 + \beta_2x_2 + \cdots + \beta_px_p\Bigg\}} \\
\Rightarrow \exp{\Bigg\{ln\bigg(\frac{P(y=1)}{1-P(y=1)}\bigg)\Bigg\}} &=& \exp{\Bigg\{\beta_0 + \beta_1x_1 + \beta_2x_2 + \cdots + \beta_px_p\Bigg\}} \\
\Rightarrow \frac{P(y=1)}{1-P(y=1)} &=& \exp{\Bigg\{\beta_0 + \beta_1x_1 + \beta_2x_2 + \cdots + \beta_px_p\Bigg\}} \\
\end{eqnarray*}
$$

A one-unit change in $X_i$ means that success is $e^{\beta_i}$ times as likely.

**Check:**
<details><summary>If I fit a logistic regression model using age to predict whether or not people voted, and $\text{logit}\big(P(\text{vote})\big) = 0.5 + 0.1(\text{age})$, how would you interpret the coefficient for age?</summary>
    
- As age increases by 1, the log-odds of someone voting increases by 0.1.
- As age increases by 1, someone is $e^{0.1}$ times as likely to vote, which is about 1.105 times as likely, or about 10.5% more likely to vote.
</details>

> Hint: Use the [np.exp](https://docs.scipy.org/doc/numpy/reference/generated/numpy.exp.html) function.

In [ ]:
# Use np.exp()


## Beyond binary classification problems.

When solving a binary classification problem in `sklearn`, `.predict_proba()` will return the probabilities for both classes. They should sum to 1.
- Suppose I have $k$ classes. How many probabilities would I expect `.predict_proba()` to return for each observation?

In the binary classification setting, the `.predict()` method in `sklearn` will use the threshold of 50% to classify observations.
- How might `.predict()` work when we have $k$ classes, where $k>2$?

In Sklearn's Logistic Regression the default probability to determine if an observation belongs or does not belong in a class is 50%. Using the `.predict_proba()` method we can get the probabilities for both classes of admittance possibilities.  First value is class `0`, second is class `1`.

How might we evaluate the performance of our classification model?

<a id='conclusion'></a>

## Conclusion

---

The goal of logistic regression is to find the best-fitting model to describe the relationship between a binary outcome and a set of independent variables. 

Logistic regression generates the coefficients — and in statsmodels the standard errors and significance levels — of a formula to predict a logit transformation of the probability that the characteristic of interest is present.

**Benefits of logistic regression include:**
- It's a classification algorithm that shares similar properties to linear regression.
- It's efficient and is by far the most common classification algorithm.
- The coefficients in a logistic regression model are interpretable (albeit somewhat complex); they represent the change in log-odds caused by the input variables.

<a id='solving-beta'></a>
## Solving For the Beta Coefficients (bonus)

---

Logistic regression minimizes the "deviance," which is similar to the residual sum of squares in linear regression, but is a more general form. 

**There's no closed-form solution to the beta coefficients like in linear regression, and the betas are found through optimization procedures.**
- We can't just do $\hat{\beta} = (X^TX)^{-1}X^Ty$!

If you're particularly interested in the math, here are two helpful resources:
- [A good blog post](http://www.win-vector.com/blog/2011/09/the-simpler-derivation-of-logistic-regression/) on the logistic regression beta coefficient derivation.
- [This paper](https://www.stat.cmu.edu/~cshalizi/402/lectures/14-logistic-regression/lecture-14.pdf) is also a good reference.

<a id='logistic'></a>
## The Logistic Function (bonus)

---

The inverse function of the logit is called the **logistic function**. 

By inverting the logit, we can have the right side of our regression equation solve explicitly for $P(y = 1)$:

### $$P(y=1) = logit^{-1}\left(\beta_0 + \sum_{j}^p\beta_jx_j\right)$$

Where:

### $$logit^{-1}(a) = logistic(a) = \frac{e^{a}}{e^{a} + 1}$$ 

Giving us:

### $$P(y=1) = \frac{e^{\left(\beta_0 + \beta_1x_1 + \beta_2x_2 + \cdots + \beta_px_p\right)}}{e^{\left(\beta_0 + \beta_1x_1 + \beta_2x_2 + \cdots + \beta_px_p\right)}+1}$$

<a id='resources'></a>

## Additional Resources

---

- [Logistic Regression Video Walk Through](https://www.youtube.com/watch?v=zAULhNrnuL4&noredirect=1)
- [Logistic Regression Walk Through](http://www.mc.vanderbilt.edu/gcrc/workshop_files/2004-11-12.pdf)
- [Logistic Regression w/ Statsmodel - Well Switching in Bangladesh](http://nbviewer.ipython.org/urls/raw.github.com/carljv/Will_it_Python/master/ARM/ch5/arsenic_wells_switching.ipynb)
- [Odds Ratio Explanation](http://www.wright.edu/~thaddeus.tarpey/ES714glm.pdf)
- [0 and 1 are not Probabilities](http://lesswrong.com/lw/mp/0_and_1_are_not_probabilities/)